In [5]:
import sys
sys.path.append('/store/carroll/repos/chess-isofit/2025/')

from data_prep import radianceH5toEnvi

import os
from spectral.io import envi
import h5py

from glob import glob

os.environ["GDAL_PAM_ENABLED"] = "NO"

In [6]:
home = '/store/carroll/col/data/2025/'

filePathToEnviProjCs = '/store/shared/ENVI6/envi61/idl/resource/pedata/predefined/EnviPEProjcsStrings.txt'

h5_dir = os.path.join(home, 'raw/L1/radianceH5/')
h5_files = glob(os.path.join(h5_dir, '*/*/*.h5'))

outDir = os.path.join(home, 'raw/L1/radianceENVI/')

In [13]:
# get unique flightlines list
fids = sorted(glob(os.path.join(home, 'raw/L1/radianceH5/*/*/*.h5')))
fids = [x.split('/')[-1].removesuffix('_radiance.h5') for x in fids]
len(fids), fids[0]

(270, 'NEON_D13_ALMO_DP1_L046-1_20250613')

In [12]:
# those are all of the fids... of those, how many rdn files have been created?
# and then, how many are actually full useful files

'NEON_D13_ALMO_DP1_L046-1_20250613'

In [ ]:
for fid in fids:
    fp = glob(os.path.join(home, f'raw/L1/radianceENVI/{fid}_rdn'))

In [15]:
fp_rdns = glob(os.path.join(home, f'raw/L1/radianceENVI/*_rdn'))
len(fp_rdns)

245

In [3]:
fp_h5 = h5_files[0]
fp_h5

'/store/carroll/col/data/2025/raw/L1/radianceH5/2025_ALMO_1/2025061314/NEON_D13_ALMO_DP1_L046-1_20250613_radiance.h5'

In [4]:
rasterNames = ['Radiance','IGM_Data','OBS_Data','GLT_Data'] # not sure what this is for yet
rasterName = rasterNames[1]
rasterName

'IGM_Data'

In [7]:
fp_out = os.path.join(outDir, os.path.basename(fp_h5).replace('radiance.h5', 'loc'))
fp_out

'/store/carroll/col/data/2025/raw/L1/radianceENVI/NEON_D13_ALMO_DP1_L046-1_20250613_loc'

In [8]:
radianceH5toEnvi.convertH5RasterToEnvi(fp_h5, rasterName, fp_out, filePathToEnviProjCs)

In [ ]:
print('done')

In [12]:
hdf5_file = h5py.File(fp_h5,'r')
sitename = str(list(hdf5_file.items())).split("'")[1]
productType = str(list(hdf5_file[sitename].items())).split("'")[1]
sitename, productType

('ALMO', 'Radiance')

In [14]:
if productType == 'Reflectance':
    productBaseLoc = hdf5_file[sitename]['Reflectance']
elif productType == 'Radiance':
    productBaseLoc = hdf5_file[sitename]['Radiance']
productBaseLoc

<HDF5 group "/ALMO/Radiance" (3 members)>

In [19]:
raster = rasterName
raster

'rdn'

In [7]:
raster, metadata = radianceH5toEnvi.h5refl2array(fp_h5, rasterName, spatialIndexesToRead = None, bandIndexesToRead = None)

KeyError: "Unable to synchronously open object (object 'Ancillary_Imagery' doesn't exist)"

In [16]:
# what is inside of a neon radiance h5 file?

fp = glob('/store/carroll/col/data/2025/raw/L1/radianceH5/2025_CRBU_2/*/*.h5')[0]

with h5py.File(fp, 'r') as f:
    print(list(f.keys()))
    tmp = f['CRBU']

['CRBU']


In [17]:
# def print_structure(name, obj):
#     print(name)

def print_structure(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(f"📄 Dataset: {name}, shape={obj.shape}, dtype={obj.dtype}")
    elif isinstance(obj, h5py.Group):
        print(f"📁 Group: {name}")

with h5py.File(fp, "r") as f:
    f.visititems(print_structure)

📁 Group: CRBU
📁 Group: CRBU/Radiance
📁 Group: CRBU/Radiance/Metadata
📁 Group: CRBU/Radiance/Metadata/Ancillary_Rasters
📄 Dataset: CRBU/Radiance/Metadata/Ancillary_Rasters/BDE, shape=(598, 426), dtype=uint8
📄 Dataset: CRBU/Radiance/Metadata/Ancillary_Rasters/GLT_Data, shape=(7741, 6104, 2), dtype=float32
📄 Dataset: CRBU/Radiance/Metadata/Ancillary_Rasters/IGM_Data, shape=(7741, 6104, 3), dtype=float32
📄 Dataset: CRBU/Radiance/Metadata/Ancillary_Rasters/OBS_Data, shape=(7741, 6104, 11), dtype=float32
📁 Group: CRBU/Radiance/Metadata/Coordinate_System
📄 Dataset: CRBU/Radiance/Metadata/Coordinate_System/Coordinate_System_String, shape=(), dtype=object
📄 Dataset: CRBU/Radiance/Metadata/Coordinate_System/EPSG Code, shape=(), dtype=object
📄 Dataset: CRBU/Radiance/Metadata/Coordinate_System/Map_Info, shape=(), dtype=object
📄 Dataset: CRBU/Radiance/Metadata/Coordinate_System/Proj4, shape=(), dtype=object
📁 Group: CRBU/Radiance/Metadata/Flight_Trajectory
📄 Dataset: CRBU/Radiance/Metadata/Flight_T

In [10]:
radianceH5toEnvi.getRasterGdalDtype()

NameError: name 'radianceH5toEnvi' is not defined